# NYC Taxi Hourly Demand Prediction
## Big Data Analytics Exam Easter 2026
### **Program:** MSc Data Science

### Team Members:
| Name | Access Number | Registration Number |
| --- | --- | --- |
| Charles MUGANGA | B35391 | S25M19/026 |
| Irene Comfort AKONGO | B34971 | S25M19/001 |

#### Environment setup and loading libraries

We configured Spark explicitly for a windows laptop

In [2]:
import os, platform
if platform.system() == "Windows":
    os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
    if r"C:\hadoop\bin" not in os.environ.get("PATH", ""):
        os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

In [3]:
import sys
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType, TimestampType,
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler,
)
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor, GBTRegressor,
    DecisionTreeRegressor,
)
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

### Reproducibility
SEED = 42
np.random.seed(SEED)

### Project paths
PROJECT_ROOT = Path(".").resolve() if Path("data").exists() else Path("..").resolve()
RAW_DIR     = PROJECT_ROOT / "data" / "raw"
PROC_DIR    = PROJECT_ROOT / "data" / "processed"
STREAM_IN   = PROJECT_ROOT / "data" / "stream_input"
STREAM_CKPT = PROJECT_ROOT / "data" / "stream_checkpoint"
STREAM_OUT  = PROJECT_ROOT / "data" / "stream_output"
MODEL_DIR   = PROJECT_ROOT / "models"
FIG_DIR     = PROJECT_ROOT / "outputs" / "figures"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

for d in (PROC_DIR, STREAM_IN, STREAM_CKPT, STREAM_OUT, MODEL_DIR, FIG_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: D:\UCU\UCU Easter 2026\Postgraduate\Big data\exam


In [4]:
spark = (
    SparkSession.builder
    .appName("UCU_BDA_NYC_Taxi_Demand")
    .master("local[*]")                                    # We are to use every core on the laptop
    .config("spark.driver.memory", "16g")                  # laptop has 40 GB \u2014 give Spark real headroom
    .config("spark.driver.maxResultSize", "4g")            # large collects (toPandas) won't trip the 1g default
    .config("spark.sql.shuffle.partitions", "16")          # default 200 is wasteful locally
    .config("spark.sql.adaptive.enabled", "true")          # AQE: shrink shuffle stages
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism)

Spark 4.1.1 | cores: 16


#### 1.Ingestion with an explicit schema

Here are two design choices worth calling out:

1. **Explicit schema**, not `inferSchema`. Inference reads the data twice fine for a notebook demo, a waste on a laptop with 10 M+ rows.
2. **One function** that works with either CSV (raw drop) or Parquet (our columnar store). The startup lands data as CSV and we promote it toParquet once, all downstream work reads Parquet.

In [5]:
TAXI_SCHEMA = StructType([
    StructField("VendorID",              IntegerType(), True),
    StructField("tpep_pickup_datetime",  TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count",       DoubleType(), True),
    StructField("trip_distance",         DoubleType(), True),
    StructField("RatecodeID",            DoubleType(), True),
    StructField("store_and_fwd_flag",    StringType(), True),
    StructField("PULocationID",          IntegerType(), True),
    StructField("DOLocationID",          IntegerType(), True),
    StructField("payment_type",          IntegerType(), True),
    StructField("fare_amount",           DoubleType(), True),
    StructField("extra",                 DoubleType(), True),
    StructField("mta_tax",               DoubleType(), True),
    StructField("tip_amount",            DoubleType(), True),
    StructField("tolls_amount",          DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount",          DoubleType(), True),
    StructField("congestion_surcharge",  DoubleType(), True),
])


def load_taxi(path: str | Path):
    """Loading NYC taxi data Parquet using an explicit schema."""
    p = str(path)
    if p.endswith(".parquet") or Path(p).is_dir():
        return spark.read.parquet(p)
    return (
        spark.read
        .option("header", True)
        .schema(TAXI_SCHEMA)
        .csv(p)
    )

parquet_files = sorted(RAW_DIR.glob("*.parquet"))
csv_files     = sorted(RAW_DIR.glob("*.csv"))
assert parquet_files or csv_files, (
    "No data found in data/raw/. Drop the NYC Taxi Parquet/CSV file(s) there "
    "or run: python src/generate_sample_data.py"
)

if parquet_files:
    print(f"Loading {len(parquet_files)} Parquet file(s):",
          [p.name for p in parquet_files])
    df_raw = spark.read.parquet(*[str(p) for p in parquet_files])
    df_raw = df_raw.select(
        F.col("VendorID").cast("int").alias("VendorID"),
        F.col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
        F.col("passenger_count").cast("double").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance"),
        F.col("RatecodeID").cast("double").alias("RatecodeID"),
        F.col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
        F.col("PULocationID").cast("int").alias("PULocationID"),
        F.col("DOLocationID").cast("int").alias("DOLocationID"),
        F.col("payment_type").cast("int").alias("payment_type"),
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("extra").cast("double").alias("extra"),
        F.col("mta_tax").cast("double").alias("mta_tax"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("tolls_amount").cast("double").alias("tolls_amount"),
        F.col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
        F.col("total_amount").cast("double").alias("total_amount"),
        F.col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
    )
else:
    RAW_PATH = str(csv_files[0])
    print("Loading CSV:", RAW_PATH)
    df_raw = load_taxi(RAW_PATH)

print("Raw rows:", df_raw.count(), "| columns:", len(df_raw.columns))
df_raw.printSchema()

Loading 3 Parquet file(s): ['yellow_tripdata_2025-02.parquet', 'yellow_tripdata_2025-03.parquet', 'yellow_tripdata_2025-12.parquet']
Raw rows: 12027806 | columns: 18
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



### 2. Data quality & EDA

Here we perform a quick inventory of quality issues, these drive the cleaning decisions in the next cell. For a startup with no dedicated data-engineering team this audit step is where most of the risk lives.

In [6]:
total = df_raw.count()
quality = df_raw.agg(
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("null_passenger"),
    F.sum(F.col("trip_distance").isNull().cast("int")).alias("null_distance"),
    F.sum(F.col("fare_amount").isNull().cast("int")).alias("null_fare"),
    F.sum((F.col("fare_amount") <= 0).cast("int")).alias("nonpositive_fare"),
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("nonpositive_distance"),
    F.sum((F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")).cast("int"))
     .alias("bad_timestamps"),
).toPandas().T.rename(columns={0: "count"})
quality["pct"] = (quality["count"] / total * 100).round(3)
print("Quality audit (total rows = {:,})".format(total))
print(quality)

Quality audit (total rows = 12,027,806)
                        count     pct
null_passenger        2919082  24.269
null_distance               0   0.000
null_fare                   0   0.000
nonpositive_fare       445190   3.701
nonpositive_distance   356149   2.961
bad_timestamps          85465   0.711


### 3. Cleaning & promotion to parquet 

**Filters (each justified):**
- Dropping rows with non-positive fare or distance refunds / data-entry errors.
- Dropping rows where drop-off precedes pickup, corrupt timestamps.
- Clip trip distance at the 99.5th percentile to tame outlier long-haul trips.
- Fill `passenger_count` nulls with the mode (1), safe, most trips are solo.
- Dropping exact duplicates.

**Why parquet?**: columnar + compressed + schema-embedded; 5-10× smaller than
CSV and enables predicate push-down. Critical when RAM is the bottleneck.

In [7]:
### percentile-based distance cap are computed on a sample for speed.
dist_cap = (
    df_raw.sample(fraction=0.1, seed=SEED)
    .approxQuantile("trip_distance", [0.995], 0.01)[0]
)
print(f"Capping trip_distance at {dist_cap:.2f} miles (99.5th percentile)")

df_clean = (
    df_raw
    .dropDuplicates()
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") <= dist_cap)
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter((F.year("tpep_pickup_datetime") >= 2009) &
            (F.year("tpep_pickup_datetime") <= 2030))
    .withColumn(
        "passenger_count",
        F.when(F.col("passenger_count").isNull(), F.lit(1.0))
         .otherwise(F.col("passenger_count")),
    )
    .withColumn(
        "trip_duration_min",
        (F.col("tpep_dropoff_datetime").cast("long")
         - F.col("tpep_pickup_datetime").cast("long")) / 60.0,
    )
    .filter((F.col("trip_duration_min") >= 1) & (F.col("trip_duration_min") <= 180))
)

clean_count = df_clean.count()
print(f"After cleaning: {clean_count:,} rows "
      f"({(clean_count/total)*100:.2f}% of raw kept).")

df_clean = df_clean.withColumn(
    "pickup_date", F.to_date("tpep_pickup_datetime")
)
clean_path = str(PROC_DIR / "trips_clean.parquet")
(df_clean.write
    .mode("overwrite")
    .partitionBy("pickup_date")
    .parquet(clean_path))
print("Clean Parquet written:", clean_path)

### reloading from Parquet so downstream uses the compact form.
df = spark.read.parquet(clean_path)
df.cache()
print("Cached clean dataframe rows:", df.count())

Capping trip_distance at 320136.29 miles (99.5th percentile)
After cleaning: 11,145,699 rows (92.67% of raw kept).
Clean Parquet written: D:\UCU\UCU Easter 2026\Postgraduate\Big data\exam\data\processed\trips_clean.parquet
Cached clean dataframe rows: 11145699


### 4. Target construction, hourly zone demand

We pivoted from *row-per-trip* to *row-per-(zone, hour)* with a `pickup_count`target. This transforms a transactional stream into the supervised-learning panel that answers the stakeholder's question.

In [8]:
df_hour = (
    df.withColumn("pickup_hour", F.date_trunc("hour", "tpep_pickup_datetime"))
      .groupBy("PULocationID", "pickup_hour")
      .agg(
          F.count("*").alias("pickup_count"),
          F.avg("trip_distance").alias("avg_distance"),
          F.avg("fare_amount").alias("avg_fare"),
          F.avg("trip_duration_min").alias("avg_duration"),
      )
)
print("Hourly-zone panel rows:", df_hour.count())
df_hour.show(5, truncate=False)

Hourly-zone panel rows: 337898
+------------+-------------------+------------+------------------+------------------+------------------+
|PULocationID|pickup_hour        |pickup_count|avg_distance      |avg_fare          |avg_duration      |
+------------+-------------------+------------+------------------+------------------+------------------+
|231         |2025-12-13 00:00:00|236         |3.273898305084747 |24.874618644067798|15.384039548022598|
|144         |2025-12-13 01:00:00|216         |2.3277314814814813|19.6774074074074  |12.807638888888889|
|234         |2025-12-13 01:00:00|259         |2.902277992277992 |20.32818532818532 |14.705598455598452|
|255         |2025-12-13 01:00:00|84          |4.47797619047619  |26.25904761904762 |17.906150793650795|
|142         |2025-12-13 02:00:00|25          |4.7943999999999996|21.785600000000002|15.699333333333334|
+------------+-------------------+------------+------------------+------------------+------------------+
only showing top 5 rows


### 5. Feature engineering

**Temporal features** (from `pickup_hour`): hour-of-day, day-of-week, weekend flag, month. Cyclical encoding is skipped because tree models handle raw integers well and we are comparing a linear baseline anyway.

**Lag features** (the real signal for time series): demand 1 hour ago, 24 hours ago (same hour yesterday), 168 hours ago (same hour last week), and a 24-hour rolling mean. These are the variables any competent forecast must beat a persistence baseline on.

In [9]:
df_feat = (
    df_hour
    .withColumn("hour",      F.hour("pickup_hour"))
    .withColumn("dow",       F.dayofweek("pickup_hour"))          # 1=Sunday
    .withColumn("is_weekend",(F.col("dow").isin(1, 7)).cast("int"))
    .withColumn("month",     F.month("pickup_hour"))
    .withColumn("day",       F.dayofmonth("pickup_hour"))
)

##lags require an ordered window *within* each zone.
w_zone = Window.partitionBy("PULocationID").orderBy("pickup_hour")
df_feat = (
    df_feat
    .withColumn("lag_1h",   F.lag("pickup_count", 1).over(w_zone))
    .withColumn("lag_24h",  F.lag("pickup_count", 24).over(w_zone))
    .withColumn("lag_168h", F.lag("pickup_count", 168).over(w_zone))
    .withColumn(
        "roll_mean_24h",
        F.avg("pickup_count").over(w_zone.rowsBetween(-24, -1)),
    )
)

### we drop the warm-up rows where lags are null (the first week of each zone).
df_feat = df_feat.dropna(subset=["lag_1h", "lag_24h", "lag_168h", "roll_mean_24h"])
print("Feature-complete rows:", df_feat.count())
df_feat.show(3, truncate=False)

Feature-complete rows: 297682
+------------+-------------------+------------+-----------------+------------------+------------------+----+---+----------+-----+---+------+-------+--------+------------------+
|PULocationID|pickup_hour        |pickup_count|avg_distance     |avg_fare          |avg_duration      |hour|dow|is_weekend|month|day|lag_1h|lag_24h|lag_168h|roll_mean_24h     |
+------------+-------------------+------------+-----------------+------------------+------------------+----+---+----------+-----+---+------+-------+--------+------------------+
|12          |2025-02-18 10:00:00|2           |3.565            |22.95             |23.983333333333334|10  |3  |0         |2    |18 |1     |4      |1       |4.0               |
|12          |2025-02-18 11:00:00|6           |3.57             |24.816666666666663|24.980555555555554|11  |3  |0         |2    |18 |2     |4      |1       |3.9166666666666665|
|12          |2025-02-18 12:00:00|5           |4.726000000000001|27.698            |2

### 6. Temporal train / validation / test split

A random split would leak future hours into the past. We use a **strictly chronological split**: 70% train, 15% validation, 15% test  which is the industry-standard protocol for any time-series forecasting study.

In [10]:
hour_long = df_feat.select(F.col("pickup_hour").cast("long").alias("ts"))
q_train, q_val = hour_long.approxQuantile("ts", [0.70, 0.85], 0.001)

t_train_end = pd.Timestamp(q_train, unit="s")
t_val_end   = pd.Timestamp(q_val,   unit="s")

t_lo, t_hi = df_feat.agg(
    F.min("pickup_hour").alias("lo"),
    F.max("pickup_hour").alias("hi"),
).collect()[0]

print(f"Full window: {t_lo}  \u2192  {t_hi}")
print(f"Train   end: {t_train_end}")
print(f"Valid   end: {t_val_end}")

train_df = df_feat.filter(F.col("pickup_hour") <  F.lit(t_train_end))
val_df   = df_feat.filter((F.col("pickup_hour") >= F.lit(t_train_end)) &
                          (F.col("pickup_hour") <  F.lit(t_val_end)))
test_df  = df_feat.filter(F.col("pickup_hour") >= F.lit(t_val_end))

n_train, n_val, n_test = train_df.count(), val_df.count(), test_df.count()
print(f"Rows \u2014 train: {n_train:,}  val: {n_val:,}  test: {n_test:,}")
assert n_val > 0 and n_test > 0, "Empty val/test split \u2014 check data coverage in data/raw/."

Full window: 2025-02-07 22:00:00  →  2025-12-31 23:00:00
Train   end: 2025-12-09 09:00:00
Valid   end: 2025-12-20 11:00:00
Rows — train: 207,609  val: 44,738  test: 45,335


### 7. ML pipeline (Spark MLlib Pipeline API)

This is a proper `Pipeline` rather than ad-hoc transforms because it <br> 
(a) guarantees the same transforms at inference time. <br> 
(b) makes it easy to swap in different estimators and compare them fairly.<br> 
(c) is the industry-standard way to productionize Spark MLlib models.

**Stages**
1. `StringIndexer` on `PULocationID` (we treat zone id as categorical).
2. `OneHotEncoder` on the indexed zone,  linear models need it, tree models tolerate it and we want a single pipeline across all three.
3. `VectorAssembler` combining all features into `features`.
4. `StandardScaler` this is only strictly needed for LR but harmless for trees.
5. The estimator.

We then train the models and pick on **validation RMSE**.

In [11]:
CAT_COLS = ["PULocationID"]
NUM_COLS = [
    "hour", "dow", "is_weekend", "month", "day",
    "avg_distance", "avg_fare", "avg_duration",
    "lag_1h", "lag_24h", "lag_168h", "roll_mean_24h",
]
TARGET = "pickup_count"

### a single indexer is shared by both pipelines.
indexer = StringIndexer(
    inputCols=CAT_COLS, outputCols=[c + "_idx" for c in CAT_COLS],
    handleInvalid="keep",                        ## for unseen zones \u2192 extra bucket
)

assembler_tree = VectorAssembler(
    inputCols=[c + "_idx" for c in CAT_COLS] + NUM_COLS,
    outputCol="features",
    handleInvalid="skip",
)

### linear pipeline, OHE + StandardScaler (LR needs both).
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in CAT_COLS],
    outputCols=[c + "_ohe" for c in CAT_COLS],
    handleInvalid="keep",
)
assembler_lin = VectorAssembler(
    inputCols=[c + "_ohe" for c in CAT_COLS] + NUM_COLS,
    outputCol="features_raw",
    handleInvalid="skip",
)
scaler = StandardScaler(inputCol="features_raw", outputCol="features",
                        withMean=False, withStd=True)

PRE_LINEAR = [indexer, encoder, assembler_lin, scaler]
PRE_TREE   = [indexer, assembler_tree]


def make_pipeline(estimator, kind="tree"):
    """Building a Pipeline with the right preprocessing for the estimator family."""
    stages = PRE_TREE if kind == "tree" else PRE_LINEAR
    return Pipeline(stages=stages + [estimator])


rmse_eval = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction",
                                metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction",
                                metricName="mae")
r2_eval   = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction",
                                metricName="r2")


def evaluate(model, df_eval, label):
    pred = model.transform(df_eval)
    return {
        "model": label,
        "rmse": rmse_eval.evaluate(pred),
        "mae":  mae_eval.evaluate(pred),
        "r2":   r2_eval.evaluate(pred),
    }

#### 7.1 Baseline: persistence (no model)
Before any ML, what does *"demand next hour = demand this hour"* score?
Any model we ship must beat this.

In [12]:
persist_pred = val_df.withColumn("prediction", F.col("lag_1h").cast("double"))
baseline_metrics = {
    "model": "persistence_lag1h",
    "rmse": rmse_eval.evaluate(persist_pred),
    "mae":  mae_eval.evaluate(persist_pred),
    "r2":   r2_eval.evaluate(persist_pred),
}
print("Baseline (val):", baseline_metrics)

Baseline (val): {'model': 'persistence_lag1h', 'rmse': 22.352441822225543, 'mae': 9.471567794715902, 'r2': 0.9212480460835748}


### 7.2 Six-model zoo: Spark MLlib + scikit-learn hybrid

| # | Model | Library | Rationale |
|---|---|---|---|
| 1 | Linear Regression       | Spark MLlib   | Linear baseline |
| 2 | Decision Tree           | Spark MLlib   | Single-tree reference — what one tree learns |
| 3 | Random Forest           | Spark MLlib   | Bagging ensemble |
| 4 | Gradient-Boosted Trees  | Spark MLlib   | Boosting ensemble |
| 5 | SVR (RBF kernel)        | scikit-learn  | Spark has no SVR; trained on a stratified sample |
| 6 | MLP Regressor (NN)      | scikit-learn  | Spark has no MLP regressor |

A **persistence baseline** (`pred = lag_1h`) is reported alongside. If it beats every ML model, that is an honest finding about how autocorrelated taxi demand is at the hourly level — we discuss it in the reflection.

**Fair comparison guarantee.** All six models see the *same* engineered features and the *same* train/val/test split. SVR and MLP consume a sampled subset (O(n²) memory in SVR's case), which we make explicit in the evaluation so the comparison is interpreted correctly.

In [13]:
lr   = LinearRegression(featuresCol="features", labelCol=TARGET,
                        maxIter=50, regParam=0.1, elasticNetParam=0.0)
dt   = DecisionTreeRegressor(featuresCol="features", labelCol=TARGET,
                             maxDepth=10, maxBins=256, seed=SEED)
rf   = RandomForestRegressor(featuresCol="features", labelCol=TARGET,
                             numTrees=60, maxDepth=10, maxBins=256, seed=SEED)
gbt  = GBTRegressor(featuresCol="features", labelCol=TARGET,
                    maxIter=60, maxDepth=6, stepSize=0.1, maxBins=256, seed=SEED)

SPARK_MODELS = [
    ("linear_regression", lr,  "linear"),
    ("decision_tree",     dt,  "tree"),
    ("random_forest",     rf,  "tree"),
    ("gbt",               gbt, "tree"),
]

models = {}           
val_scores = []

for name, est, kind in SPARK_MODELS:
    print(f"\nTraining {name} (Spark) …")
    pipe = make_pipeline(est, kind=kind)
    m = pipe.fit(train_df)
    models[name] = m
    val_scores.append(evaluate(m, val_df, name))


Training linear_regression (Spark) …

Training decision_tree (Spark) …

Training random_forest (Spark) …

Training gbt (Spark) …


### 7.2b Sklearn models: SVR and MLP

Now we pull the *already-engineered* train/val/test DataFrames into pandas via Arrow. The numeric feature columns alone are used. Trees handled zone ID via OHE just fine because they
partition, kernel and neural models prefer a **compact dense numeric matrix**, so we pass zone as an integer feature rather than OHE standard practice for small-data MLP/SVR.

In [14]:
SKLEARN_FEATURES = ["PULocationID"] + NUM_COLS            ### having zone as int, not OHE

def to_pandas_xy(spark_df, sample_frac=None):
    sdf = spark_df.select(*SKLEARN_FEATURES, TARGET)
    if sample_frac is not None and sample_frac < 1.0:
        sdf = sdf.sample(fraction=sample_frac, seed=SEED)
    pdf = sdf.toPandas().dropna()
    X = pdf[SKLEARN_FEATURES].to_numpy(dtype="float32")
    y = pdf[TARGET].to_numpy(dtype="float32")
    return X, y


SVR_TRAIN_CAP = 30_000
svr_frac = min(1.0, SVR_TRAIN_CAP / max(1, train_df.count()))

MLP_TRAIN_CAP = 200_000
mlp_frac = min(1.0, MLP_TRAIN_CAP / max(1, train_df.count()))

print(f"Preparing sklearn training data... \n "
      f"SVR sample fraction {svr_frac:.3f}, MLP sample fraction {mlp_frac:.3f}")

X_train_svr, y_train_svr = to_pandas_xy(train_df, sample_frac=svr_frac)
X_train_mlp, y_train_mlp = to_pandas_xy(train_df, sample_frac=mlp_frac)
X_val,  y_val  = to_pandas_xy(val_df)
X_test, y_test = to_pandas_xy(test_df)

print(f"Shapes: SVR train: {X_train_svr.shape}, MLP train: {X_train_mlp.shape}, "
      f"val: {X_val.shape}, test: {X_test.shape}")

from sklearn.preprocessing import StandardScaler as SkStandardScaler
scaler_sk = SkStandardScaler()
X_train_svr_s = scaler_sk.fit_transform(X_train_svr)
X_val_svr_s   = scaler_sk.transform(X_val)
X_test_svr_s  = scaler_sk.transform(X_test)

scaler_mlp = SkStandardScaler()
X_train_mlp_s = scaler_mlp.fit_transform(X_train_mlp)
X_val_mlp_s   = scaler_mlp.transform(X_val)
X_test_mlp_s  = scaler_mlp.transform(X_test)

### SVR
print("\nTraining svr_rbf (sklearn) …")
svr = SVR(kernel="rbf", C=1.0, gamma="scale", cache_size=500)
svr.fit(X_train_svr_s, y_train_svr)
val_pred_svr = svr.predict(X_val_svr_s)
val_scores.append({
    "model": "svr_rbf",
    "rmse":  float(np.sqrt(mean_squared_error(y_val, val_pred_svr))),
    "mae":   float(mean_absolute_error(y_val, val_pred_svr)),
    "r2":    float(r2_score(y_val, val_pred_svr)),
})
models["svr_rbf"] = ("sklearn", svr, scaler_sk)

### MLP
print("Training mlp_regressor (sklearn) …")
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,                     ## L2
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,            ### val split inside sklearn
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=False,
)
mlp.fit(X_train_mlp_s, y_train_mlp)
val_pred_mlp = mlp.predict(X_val_mlp_s)
val_scores.append({
    "model": "mlp_regressor",
    "rmse":  float(np.sqrt(mean_squared_error(y_val, val_pred_mlp))),
    "mae":   float(mean_absolute_error(y_val, val_pred_mlp)),
    "r2":    float(r2_score(y_val, val_pred_mlp)),
})
models["mlp_regressor"] = ("sklearn", mlp, scaler_mlp)
print(f"  MLP converged in {mlp.n_iter_} iterations.")

Preparing sklearn training data... 
 SVR sample fraction 0.145, MLP sample fraction 0.963
Shapes: SVR train: (30027, 13), MLP train: (199872, 13), val: (44738, 13), test: (45335, 13)

Training svr_rbf (sklearn) …
Training mlp_regressor (sklearn) …
  MLP converged in 112 iterations.


In [15]:
val_scores.append(baseline_metrics)
val_df_pd = pd.DataFrame(val_scores).sort_values("rmse").reset_index(drop=True)
print("\nValidation leaderboard:")
print(val_df_pd.round(3).to_string(index=False))
val_df_pd.to_csv(METRICS_DIR / "val_leaderboard.csv", index=False)


Validation leaderboard:
            model   rmse    mae    r2
    random_forest 13.823  6.102 0.970
    mlp_regressor 13.835  6.637 0.970
              gbt 14.207  6.386 0.968
linear_regression 16.734  7.976 0.956
    decision_tree 17.195  7.107 0.953
persistence_lag1h 22.352  9.472 0.921
          svr_rbf 42.733 12.698 0.712


#### 7.3. Test-set evaluation of the winner

We pick the winner by validation RMSE. If the persistence baseline wins, that is a legitimate and reportable result: it means the hourly autocorrelation in taxi demand is so strong that our current feature set cannot improve on "same as last hour." This is a common finding in short-horizon demand forecasting and motivates the richer features (weather, events, zone adjacency) that we discuss in the reflection.

We still save the **best ML model** for deployment, because persistence is not deployable as a *model* it is an identity function — and an ML model provides value during regime changes (morning ramp-up, late-night drop-off) even when its aggregate metric is slightly worse.

In [16]:
def evaluate_any(model_entry, spark_df, X_np, y_np, label):
    """the unified evaluator across Spark PipelineModels and sklearn estimators."""
    if isinstance(model_entry, tuple) and model_entry[0] == "sklearn":
        _, est, sk_scaler = model_entry
        pred = est.predict(sk_scaler.transform(X_np))
        return {
            "model": label,
            "rmse":  float(np.sqrt(mean_squared_error(y_np, pred))),
            "mae":   float(mean_absolute_error(y_np, pred)),
            "r2":    float(r2_score(y_np, pred)),
        }
    ### the Spark PipelineModel
    pred_df = model_entry.transform(spark_df)
    return {
        "model": label,
        "rmse":  rmse_eval.evaluate(pred_df),
        "mae":   mae_eval.evaluate(pred_df),
        "r2":    r2_eval.evaluate(pred_df),
    }

winner_name = val_df_pd.iloc[0]["model"]
print(f"Overall winner (lowest val RMSE): {winner_name}")

### best *ML* model.
ml_leaderboard = val_df_pd[val_df_pd["model"] != "persistence_lag1h"].reset_index(drop=True)
best_ml_name = ml_leaderboard.iloc[0]["model"]
best_ml_model = models[best_ml_name]
print(f"Best ML model (for deployment): {best_ml_name}")

### test-set scores for ALL models with full transparency.
test_scores = []
for name, m in models.items():
    test_scores.append(evaluate_any(m, test_df, X_test, y_test, f"{name}_TEST"))
test_scores.append({
    "model": "persistence_lag1h_TEST",
    "rmse": rmse_eval.evaluate(test_df.withColumn("prediction", F.col("lag_1h").cast("double"))),
    "mae":  mae_eval.evaluate(test_df.withColumn("prediction", F.col("lag_1h").cast("double"))),
    "r2":   r2_eval.evaluate(test_df.withColumn("prediction", F.col("lag_1h").cast("double"))),
})
test_df_pd = pd.DataFrame(test_scores).sort_values("rmse").reset_index(drop=True)
print("\nTest-set leaderboard:")
print(test_df_pd.round(3).to_string(index=False))
test_df_pd.to_csv(METRICS_DIR / "test_scores.csv", index=False)

### flagging for the narrative
persistence_test_rmse = next(s["rmse"] for s in test_scores
                              if s["model"] == "persistence_lag1h_TEST")
best_ml_test_rmse = next(s["rmse"] for s in test_scores
                          if s["model"] == f"{best_ml_name}_TEST")
ml_beats_persistence = best_ml_test_rmse < persistence_test_rmse
gap_pct = (persistence_test_rmse - best_ml_test_rmse) / persistence_test_rmse * 100
print(f"\n-> Best ML ({best_ml_name}) vs persistence on test: "
      f"{'BEATS baseline' if ml_beats_persistence else 'DOES NOT beat baseline'} "
      f"({gap_pct:+.2f}% RMSE)")

Overall winner (lowest val RMSE): random_forest
Best ML model (for deployment): random_forest

Test-set leaderboard:
                 model   rmse    mae    r2
    random_forest_TEST 13.313  5.717 0.934
    mlp_regressor_TEST 13.751  6.690 0.929
persistence_lag1h_TEST 14.353  6.264 0.923
linear_regression_TEST 14.449  6.946 0.922
              gbt_TEST 14.829  6.341 0.918
    decision_tree_TEST 15.899  6.422 0.905
          svr_rbf_TEST 23.674 12.677 0.790

-> Best ML (random_forest) vs persistence on test: BEATS baseline (+7.24% RMSE)


#### 7.4. Persist the trained pipeline
Saving the whole `PipelineModel` (not just the estimator) is what makes the offline -> online hand-off safe: the serving code loads exactly the same transformers that saw training data.

In [ ]:
import pickle

model_path = str(MODEL_DIR / f"pipeline_{best_ml_name}")
if isinstance(best_ml_model, tuple) and best_ml_model[0] == "sklearn":
    if Path(model_path).exists():
        shutil.rmtree(model_path, ignore_errors=True)
    Path(model_path).mkdir(parents=True, exist_ok=True)
    with open(Path(model_path) / "sklearn_bundle.pkl", "wb") as f:
        pickle.dump({"estimator": best_ml_model[1],
                     "scaler":    best_ml_model[2],
                     "features":  SKLEARN_FEATURES}, f)
    print(f"Saved sklearn bundle to: {model_path}")
else:
    if Path(model_path).exists():
        shutil.rmtree(model_path)
    best_ml_model.write().overwrite().save(model_path)
    print("Saved Spark pipeline to:", model_path)

Saved Spark pipeline to: D:\UCU\UCU Easter 2026\Postgraduate\Big data\exam\models\pipeline_random_forest


### 8. Batch inference demo

Loading the saved pipeline and re-score the most recent day, this is what a nightly cron job on the office laptop would do.

In [ ]:
from pyspark.ml import PipelineModel

last_day_start = t_hi - pd.Timedelta(days=1)
recent = df_feat.filter(F.col("pickup_hour") >= F.lit(last_day_start))

if isinstance(best_ml_model, tuple) and best_ml_model[0] == "sklearn":
    _, est, sk_scaler = best_ml_model
    recent_pdf = recent.select(
        "PULocationID", "pickup_hour", TARGET, *SKLEARN_FEATURES
    ).toPandas()
    X_recent = sk_scaler.transform(recent_pdf[SKLEARN_FEATURES].to_numpy("float32"))
    recent_pdf["predicted_pickups"] = est.predict(X_recent).round(2)
    preds_out = recent_pdf[["PULocationID", "pickup_hour", TARGET, "predicted_pickups"]]
    print(preds_out.head(10).to_string(index=False))
    preds_out.to_parquet(PROC_DIR / "latest_batch_predictions.parquet", index=False)
else:
    loaded = PipelineModel.load(model_path)
    preds = (
        loaded.transform(recent)
        .select("PULocationID", "pickup_hour", TARGET,
                F.round("prediction", 2).alias("predicted_pickups"))
    )
    preds.orderBy("pickup_hour", "PULocationID").show(10, truncate=False)
    preds.coalesce(1).write.mode("overwrite").parquet(
        str(PROC_DIR / "latest_batch_predictions.parquet")
    )

+------------+-------------------+------------+-----------------+
|PULocationID|pickup_hour        |pickup_count|predicted_pickups|
+------------+-------------------+------------+-----------------+
|4           |2025-12-30 23:00:00|20          |18.47            |
|7           |2025-12-30 23:00:00|8           |9.6              |
|10          |2025-12-30 23:00:00|3           |2.37             |
|13          |2025-12-30 23:00:00|4           |16.28            |
|14          |2025-12-30 23:00:00|1           |2.22             |
|15          |2025-12-30 23:00:00|1           |1.62             |
|17          |2025-12-30 23:00:00|4           |3.38             |
|22          |2025-12-30 23:00:00|1           |1.64             |
|24          |2025-12-30 23:00:00|6           |13.43            |
|25          |2025-12-30 23:00:00|8           |6.3              |
+------------+-------------------+------------+-----------------+
only showing top 10 rows


### 9. Streaming simulation (Spark Structured Streaming)

**Why we used simulated streaming?** Kafka/Kinesis is off the table — no cloud budget, intermittent internet. The realistic analogue on-prem is the`file source`, the dispatch system writes a trip-log CSV chunk to a watched folder; Spark picks it up, aggregates hourly pickups, and appends to a Parquet sink that the Ops dashboard reads.

This gives us *micro-batch* latency (seconds) without any extra infrastructure — and the same PySpark code would scale to Kafka by changing one `readStream.format(...)` line.

In [ ]:
### Splitting a slice of clean data into three "arrival" chunks.
for f in STREAM_IN.glob("*.csv"):
    f.unlink()
for f in STREAM_CKPT.glob("**/*"):
    if f.is_file():
        f.unlink()
if STREAM_OUT.exists():
    shutil.rmtree(STREAM_OUT)
STREAM_OUT.mkdir(parents=True, exist_ok=True)

stream_sample = (
    df.orderBy("tpep_pickup_datetime")
      .limit(30_000)
      .select("tpep_pickup_datetime", "PULocationID", "fare_amount", "trip_distance")
      .toPandas()
)
chunk_size = len(stream_sample) // 3 + 1
chunks = [stream_sample.iloc[i : i + chunk_size] for i in range(0, len(stream_sample), chunk_size)]
for i, ch in enumerate(chunks):
    ch.to_csv(STREAM_IN / f"arrival_{i:02d}.csv", index=False)
print("Seeded stream input:", [p.name for p in STREAM_IN.glob('*.csv')])

Seeded stream input: ['arrival_00.csv', 'arrival_01.csv', 'arrival_02.csv']


In [ ]:
import shutil, time
for d in (STREAM_CKPT, STREAM_OUT):
    if Path(d).exists():
        shutil.rmtree(d, ignore_errors=True)
    Path(d).mkdir(parents=True, exist_ok=True)

stream_schema = StructType([
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("PULocationID",         IntegerType(),  True),
    StructField("fare_amount",          DoubleType(),   True),
    StructField("trip_distance",        DoubleType(),   True),
])

stream_in = (
    spark.readStream
    .option("header", True)
    .schema(stream_schema)
    .csv(str(STREAM_IN))
)

agg_stream = (
    stream_in
    .withWatermark("tpep_pickup_datetime", "2 hours")
    .groupBy(
        F.window("tpep_pickup_datetime", "1 hour"),
        F.col("PULocationID"),
    )
    .agg(F.count("*").alias("pickup_count"),
         F.avg("fare_amount").alias("avg_fare"))
)

query = (
    agg_stream.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", str(STREAM_OUT))
    .option("checkpointLocation", str(STREAM_CKPT))
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(20)                               ### letting three micro-batches fire
query.processAllAvailable()
query.stop()
print("Streaming query stopped. Output Parquet fragments:",
      len(list(STREAM_OUT.rglob('*.parquet'))))

stream_result = spark.read.parquet(str(STREAM_OUT))
print("Streaming aggregate rows:", stream_result.count())
stream_result.orderBy("window").show(5, truncate=False)

Streaming query stopped. Output Parquet fragments: 17
Streaming aggregate rows: 961
+------------------------------------------+------------+------------+--------+
|window                                    |PULocationID|pickup_count|avg_fare|
+------------------------------------------+------------+------------+--------+
|{2009-01-01 00:00:00, 2009-01-01 01:00:00}|138         |1           |52.7    |
|{2025-01-31 22:00:00, 2025-01-31 23:00:00}|48          |1           |10.0    |
|{2025-01-31 22:00:00, 2025-01-31 23:00:00}|68          |1           |7.9     |
|{2025-01-31 23:00:00, 2025-02-01 00:00:00}|68          |1           |21.2    |
|{2025-01-31 23:00:00, 2025-02-01 00:00:00}|50          |1           |40.1    |
+------------------------------------------+------------+------------+--------+
only showing top 5 rows


### 10. Insights & visualizations for the Operations Manager

Four plots, each answering a decision the Ops Manager actually makes.

In [ ]:
hourly = (
    df_feat.groupBy("hour").agg(F.avg("pickup_count").alias("avg_pickups"))
    .orderBy("hour").toPandas()
)
dow = (
    df_feat.groupBy("dow").agg(F.avg("pickup_count").alias("avg_pickups"))
    .orderBy("dow").toPandas()
)
top_zones = (
    df_feat.groupBy("PULocationID")
    .agg(F.sum("pickup_count").alias("total_pickups"))
    .orderBy(F.desc("total_pickups")).limit(15).toPandas()
)

### Figure 1: demand by hour of day
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(hourly["hour"], hourly["avg_pickups"], color="#2E6F95")
ax.set_title("Average pickups per zone by hour of day")
ax.set_xlabel("Hour of day"); ax.set_ylabel("Avg pickups / zone / hour")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig(FIG_DIR / "01_demand_by_hour.png", dpi=150); plt.close()

### Figure 2: demand by day of week
dow_labels = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([dow_labels[int(d) - 1] for d in dow["dow"]], dow["avg_pickups"],
       color="#E1812C")
ax.set_title("Average pickups per zone by day of week")
ax.set_ylabel("Avg pickups / zone / hour")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_demand_by_dow.png", dpi=150); plt.close()

### Figure 3: top 15 pickup zones
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(top_zones["PULocationID"].astype(str), top_zones["total_pickups"],
       color="#3A923A")
ax.set_title("Top 15 pickup zones by total demand (train+val+test window)")
ax.set_xlabel("Pickup Zone ID"); ax.set_ylabel("Total pickups")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_top_zones.png", dpi=150); plt.close()

### Figure 4: predicted vs actual on test set
if isinstance(best_ml_model, tuple) and best_ml_model[0] == "sklearn":
    _, est, sk_scaler = best_ml_model
    sample_idx = np.random.RandomState(SEED).choice(
        len(X_test), size=min(5000, len(X_test)), replace=False
    )
    y_s = y_test[sample_idx]
    p_s = est.predict(sk_scaler.transform(X_test[sample_idx]))
    pred_pdf = pd.DataFrame({TARGET: y_s, "prediction": p_s})
else:
    pred_pdf = (
        best_ml_model.transform(test_df)
        .select(TARGET, "prediction").sample(fraction=0.2, seed=SEED).toPandas()
    )

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pred_pdf[TARGET], pred_pdf["prediction"], alpha=0.3, s=10)
lims = [0, max(pred_pdf[TARGET].max(), pred_pdf["prediction"].max())]
ax.plot(lims, lims, "r--", linewidth=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual pickups"); ax.set_ylabel("Predicted pickups")
ax.set_title(f"{best_ml_name} — predicted vs actual (test set)")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_pred_vs_actual.png", dpi=150); plt.close()

print("Saved figures to", FIG_DIR)

Saved figures to D:\UCU\UCU Easter 2026\Postgraduate\Big data\exam\outputs\figures


#### Feature importance for tree-based models only

In [ ]:
try:
    if isinstance(best_ml_model, tuple) and best_ml_model[0] == "sklearn":
        est = best_ml_model[1]
        if hasattr(est, "feature_importances_"):         # unlikely sklearn trees
            imps = est.feature_importances_
            feat_names = SKLEARN_FEATURES
        elif hasattr(est, "coef_"):                       # linear-ish sklearn
            imps = np.abs(np.ravel(est.coef_))
            feat_names = SKLEARN_FEATURES
        else:
            imps = None
            print(f"Feature importance not directly available for "
                  f"{best_ml_name} (use permutation importance for SVR/MLP).")
        if imps is not None:
            imp_df = (pd.DataFrame({"feature": feat_names, "imp": imps})
                        .sort_values("imp", ascending=False).head(15))
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(imp_df["feature"][::-1], imp_df["imp"][::-1], color="#7A4FB8")
            ax.set_title(f"Feature importances — {best_ml_name}")
            plt.tight_layout()
            plt.savefig(FIG_DIR / "05_feature_importance.png", dpi=150); plt.close()
            imp_df.to_csv(METRICS_DIR / "feature_importance.csv", index=False)
            print(imp_df.to_string(index=False))
    else:
        last_stage = best_ml_model.stages[-1]
        if hasattr(last_stage, "featureImportances"):
            ohe_size = best_ml_model.stages[1].categorySizes[0]
            feat_names = [f"zone_ohe_{i}" for i in range(ohe_size)] + NUM_COLS
            imps = last_stage.featureImportances.toArray()
            imp_df = (pd.DataFrame({"feature": feat_names[:len(imps)], "imp": imps})
                        .sort_values("imp", ascending=False).head(15))
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(imp_df["feature"][::-1], imp_df["imp"][::-1], color="#7A4FB8")
            ax.set_title(f"Top-15 feature importances — {best_ml_name}")
            plt.tight_layout()
            plt.savefig(FIG_DIR / "05_feature_importance.png", dpi=150); plt.close()
            imp_df.to_csv(METRICS_DIR / "feature_importance.csv", index=False)
            print(imp_df.to_string(index=False))
        else:
            print(f"Feature importance not available for {best_ml_name}.")
except Exception as e:
    print("Feature importance skipped:", e)

Feature importance skipped: 'VectorAssembler' object has no attribute 'categorySizes'


## 11 · Save the final metrics bundle

In [40]:
summary = {
    "overall_winner": winner_name,
    "best_ml_model":  best_ml_name,
    "ml_beats_persistence_on_test": bool(ml_beats_persistence),
    "best_ml_vs_persistence_test_rmse_delta_pct": round(gap_pct, 3),
    "validation": val_df_pd.to_dict(orient="records"),
    "test": test_df_pd.to_dict(orient="records"),
    "train_window":  [str(t_lo), str(t_train_end)],
    "val_window":    [str(t_train_end), str(t_val_end)],
    "test_window":   [str(t_val_end), str(t_hi)],
    "rows": {
        "raw": total,
        "clean": clean_count,
        "hourly_panel": df_feat.count(),
    },
}
with open(METRICS_DIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))

{
  "overall_winner": "random_forest",
  "best_ml_model": "random_forest",
  "ml_beats_persistence_on_test": true,
  "best_ml_vs_persistence_test_rmse_delta_pct": 7.244,
  "validation": [
    {
      "model": "random_forest",
      "rmse": 13.822553154828334,
      "mae": 6.102411872709743,
      "r2": 0.9698846490293814
    },
    {
      "model": "mlp_regressor",
      "rmse": 13.834626164212592,
      "mae": 6.6366071701049805,
      "r2": 0.9698320031166077
    },
    {
      "model": "gbt",
      "rmse": 14.206670531125772,
      "mae": 6.385575572590843,
      "r2": 0.9681876311627227
    },
    {
      "model": "linear_regression",
      "rmse": 16.734093491979955,
      "mae": 7.976167093624631,
      "r2": 0.9558616825824428
    },
    {
      "model": "decision_tree",
      "rmse": 17.194709992083745,
      "mae": 7.106746618505471,
      "r2": 0.9533983705575109
    },
    {
      "model": "persistence_lag1h",
      "rmse": 22.352441822225543,
      "mae": 9.471567794715902,

In [41]:
spark.stop()
print("Pipeline complete. All artefacts written under `outputs/` and `models/`.")

Pipeline complete. All artefacts written under `outputs/` and `models/`.
